# Practical 10 — Portfolio Optimization and Quantitative Trading

This is the Colab-friendly counterpart to the agentic Chapter 10 project. The original
practical is driven from Claude Code / Cline with a `/portfolio` command; here we call the
same deterministic reference tools in `tools/` directly so the whole **views → optimize →
backtest** pipeline runs end to end, fully offline, on the bundled five-asset universe.

The tools do all the algebra: `tools/views.py` turns four short analyst notes into a
view-adjusted expected-returns vector, `tools/optimize.py` solves the closed-form
mean-variance weights, and `tools/backtest.py` applies those weights to a 24-month return
series. In the real project the LLM only *chooses* the objective and *interprets* the
numbers — it never computes them. We reproduce that here, in plain Python.

In [ ]:
# Colab: run once to install this practical's dependencies.
# (Locally, skip if you ran `pip install -r requirements.txt`.)
%pip install numpy

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "10-portfolio-quant-trading"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## The data

Everything is a fictional, offline universe in `data/`. The market block holds five assets
(AURUM, BOREALIS, CYGNUS, DELPHI, EQUINOX) with equilibrium expected returns, a 5×5
covariance matrix, and a 24-month return series. `data/views/` holds four short analyst
notes. Let's load the market and inspect its shape.

In [ ]:
import json
import numpy as np
from tools._common import load_market, load_views

market = load_market()
print("Assets           :", market["assets"])
print("Frequency        :", market["frequency"], f"({market['periods_per_year']} periods/yr)")
print("Equilibrium E[r] :", np.round(market["expected_returns"], 4).tolist())
print("Covariance shape :", market["covariance"].shape)
print("Returns shape    :", market["returns"].shape, "(months x assets)")

## Step 1 — Views: text notes → an expected-returns vector

The agent never invents numbers from prose; `tools/views.py` does it deterministically.
It scans each note for sentiment keywords (bullish/overweight/upside → +0.02 each;
bearish/underweight/downside → −0.02 each; neutral → 0) and tilts the named asset's
equilibrium return by that amount. Same notes in, same vector out.

First, the raw analyst notes:

In [ ]:
views = load_views()
for name, text in views.items():
    print(f"--- {name} ---")
    print(text.strip())
    print()

Now run the tool. `build_views` returns `(mu, breakdown)`: the tilted vector plus the
per-view tilt detail. Note there are only four notes for five assets — CYGNUS has no note,
so it keeps its equilibrium return.

In [ ]:
from tools.views import build_views

mu, breakdown = build_views(market)
print("Asset      base E[r]   view E[r]")
for a, b, m in zip(market["assets"], market["expected_returns"], mu):
    print(f"  {a:<9} {b:+.4f}    {m:+.4f}")
print()
print("Applied tilts:")
print(json.dumps(breakdown, indent=2))

AURUM and DELPHI get lifted (bullish), BOREALIS gets cut by 0.10 (bearish, five negative
keywords), EQUINOX stays put (neutral), CYGNUS is untouched. This `mu` is what `max-sharpe`
will use; `min-variance` ignores it entirely.

## Step 2 — Optimize: closed-form mean-variance weights

`tools/optimize.py` gives two fully-invested objectives (weights sum to 1), both analytic
(NumPy `linalg.solve`, no solver, no randomness):

- **max-sharpe** — tangency portfolio `w ∝ Σ⁻¹(μ − r_f)`, using the *view-adjusted* μ.
- **min-variance** — global min-variance `w ∝ Σ⁻¹·1`, ignoring expected returns.

We call `optimize(objective, market, rf)` (which re-builds the views internally) and then
`portfolio_stats` for E[r], volatility, and Sharpe.

In [ ]:
from tools.optimize import optimize, portfolio_stats

def show_optimize(objective, rf=0.0):
    w, mu_used, Sigma = optimize(objective, market, rf=rf)
    stats = portfolio_stats(w, mu_used, Sigma, rf=rf)
    payload = {
        "objective": objective,
        "rf": rf,
        "weights": {a: round(float(x), 4) for a, x in zip(market["assets"], w)},
        "weights_sum": round(float(w.sum()), 6),
        "expected_return": round(stats["expected_return"], 4),
        "volatility": round(stats["volatility"], 4),
        "sharpe": round(stats["sharpe"], 4),
    }
    print(json.dumps(payload, indent=2))
    return w

w_sharpe = show_optimize("max-sharpe", rf=0.0)

The tangency book tilts long toward the bullish names (AURUM ~0.46, DELPHI ~0.28) and
**shorts** the bearish one (BOREALIS ~ −0.19). A negative weight is a real short position —
the tools report it as-is and we do not clip it.

Now the min-variance portfolio for contrast:

In [ ]:
w_minvar = show_optimize("min-variance")

**Try-it / interpret (from the README's "things to try").** Compare the two weight vectors:
the views move one objective but not the other. `max-sharpe` reads the analyst notes and
concentrates in AURUM/DELPHI while shorting BOREALIS; `min-variance` ignores the notes
entirely and parks the bulk of the book (~0.62) in EQUINOX, the low-volatility anchor.

In [ ]:
print(f"{'Asset':<9}{'max-sharpe':>12}{'min-variance':>14}")
for a, ws, wv in zip(market["assets"], w_sharpe, w_minvar):
    print(f"{a:<9}{ws:>12.4f}{wv:>14.4f}")

## Step 3 — Backtest: apply the weights to the return series

`tools/backtest.py` runs a buy-and-hold backtest: per-period portfolio return `R_t·w`,
cumulative `∏(1+p_t)−1`, and an annualised Sharpe `mean/std·√12`. No rebalancing, no costs.
This is **in-sample on fictional data — illustrative, not a performance claim.**

In [ ]:
from tools.backtest import backtest

def show_backtest(label, weights):
    res = backtest(weights, market["returns"], market["periods_per_year"])
    print(f"{label}:")
    print(f"  periods           = {res['periods']}")
    print(f"  cumulative_return = {res['cumulative_return']:+.4f}")
    print(f"  mean_period_return= {res['mean_period_return']:+.4f}")
    print(f"  volatility        = {res['volatility']:.4f}")
    print(f"  sharpe (annual)   = {res['sharpe']:+.4f}")
    return res

_ = show_backtest("max-sharpe", w_sharpe)
print()
_ = show_backtest("min-variance", w_minvar)

## Try-it: an explicit equal-weight book

The README suggests backtesting a naive equal-weight (0.2 each) book and comparing its
Sharpe to the optimized ones. We pass explicit weights straight to `backtest`.

In [ ]:
w_equal = np.full(len(market["assets"]), 0.2)
_ = show_backtest("equal-weight (0.2 each)", w_equal)

On this short, fictional 24-month window the results diverge sharply: the low-volatility
**min-variance** book actually posts a positive cumulative return and the best Sharpe,
while the view-driven **max-sharpe** and naive **equal-weight** books are slightly
negative. That is a reminder that an *optimizer* maximises the modelled objective
(forward-looking E[r] and Σ), not realised in-sample performance — and that on noisy data
the variance-only portfolio can be the more robust one. The exercise is the pipeline, not
the P&L: this is in-sample on invented data, not a performance claim.

## Try-it: raise the risk-free rate and watch the tangency portfolio rotate

With a higher `r_f`, the tangency portfolio `w ∝ Σ⁻¹(μ − r_f)` rotates toward the
higher-expected-return assets. Compare `rf=0.0` to `rf=0.06`.

In [ ]:
print(f"{'Asset':<9}{'rf=0.00':>10}{'rf=0.06':>10}")
w_rf0, _, _ = optimize("max-sharpe", market, rf=0.0)
w_rf6, _, _ = optimize("max-sharpe", market, rf=0.06)
for a, a0, a6 in zip(market["assets"], w_rf0, w_rf6):
    print(f"{a:<9}{a0:>10.4f}{a6:>10.4f}")

## Recap / next steps

We ran the full Chapter 10 pipeline offline, calling the same reference tools the agentic
project drives:

1. **Views** (`tools/views.py` · `build_views`) — four text notes → a view-adjusted μ.
2. **Optimize** (`tools/optimize.py` · `optimize`, `portfolio_stats`) — closed-form
   max-sharpe (tangency, uses the views) and min-variance (ignores them) weights.
3. **Backtest** (`tools/backtest.py` · `backtest`) — buy-and-hold cumulative return and
   annualised Sharpe on the 24-month series, including an equal-weight comparison.

Things to explore next:
- Edit a note in `data/views/` from "bullish" to "bearish" and re-run `build_views` —
  watch that asset's `max-sharpe` weight flip from long to short.
- Sweep `r_f` further and trace how the tangency weights migrate toward high-μ assets.
- In the full agentic repo, run `/portfolio max-sharpe` in Claude Code / Cline to have the
  agent orchestrate these same tools and write a cited report to `reports/`.
- Validate the tools with `python -m pytest -q` (fully offline).